<h1><center>How to write a scikit-learn compatible estimator</center></h1>
<h2><center>in the modern age!</center></h2>
<h3><center>Adrin Jalali</center></h3>
<h4><center>github.com/adrinjalali/talks</center></h3>
<h4><center>@probabl.ai</center></h3>
<h4><center>May 2025</center></h3>

### Intro to scikit-learn

A machine learning library implementing statistical machine learning methods.

__Important components in the API:__

- estimators (transformers and predictors)
- scorers
- meta-estimators
    - pipeline
    - grid search

``` python
# A common workflow includes a pipeline once the data is loaded.
# We usually preprocess the data and prepare it for the
# final classifier or regressor.
# We call each step an "estimator", the preprocessing steps which
# augment the data are "transformers", and the final step is a
# classifier or a regressor.
pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

# Each step can be tuned with many hyper-parameters, and we want to
# find the best hyper-parameter set for the given dataset.
parameters = {
    'vect__max_df': (0.5, 0.75, 1.0),
    'vect__ngram_range': ((1, 1), (1, 2)),
    'clf__max_iter': (20,),
    'clf__alpha': (0.00001, 0.000001),
    'clf__penalty': ('l2', 'elasticnet'),
}

# find the best parameters for both the feature extraction and the
# classifier, we use a grid search.
grid_search = GridSearchCV(pipeline, parameters)
```

But it can be a more complicated pipeline!
See the full example [here](https://scikit-learn.org/dev/auto_examples/compose/plot_column_transformer.html).
```python
pipeline = Pipeline(
    [
        # Extract subject & body
        ("subjectbody", subject_body_transformer),
        # Use ColumnTransformer to combine the subject and body features
        (
            "union",
            ColumnTransformer(
                [
                    # bag-of-words for subject (col 0)
                    ("subject", TfidfVectorizer(min_df=50), 0),
                    # bag-of-words with decomposition for body (col 1)
                    (
                        "body_bow",
                        Pipeline(
                            [
                                ("tfidf", TfidfVectorizer()),
                                ("best", TruncatedSVD(n_components=50)),
                            ]
                        ),
                        1,
                    ),
                    # Pipeline for pulling text stats from post's body
                    (
                        "body_stats",
                        Pipeline(
                            [
                                (
                                    "stats",
                                    text_stats_transformer,
                                ),  # returns a list of dicts
                                (
                                    "vect",
                                    DictVectorizer(),
                                ),  # list of dicts -> feature matrix
                            ]
                        ),
                        1,
                    ),
                ],
                # weight above ColumnTransformer features
                transformer_weights={
                    "subject": 0.8,
                    "body_bow": 0.5,
                    "body_stats": 1.0,
                },
            ),
        ),
        # Use a SVC classifier on the combined features
        ("svc", LinearSVC(dual=False)),
    ],
    verbose=True,
)
```

## Why a custom estimator?

- scikit-learn doesn't include all algorithms, and it has a very high bar for including one. You can test your new or modified algorithm as a custom estimator.
- The library does not include methods which are appropriate only for a small set of use-cases, and if you happen to work on one of those problems, you might need to develop your own estimator to tackle the specific issues you have.
- You may want to add some steps before or after running each step, in which case you can write a meta-estimator wrapping around the steps you usually would have in a pipeline.

### Basic Estimator API

- `fit (X, y[, sample_weight], **kwargs)`
- `predict(X)` (`predict_proba` and `decision_function`)
- `transform(X)`
- `score(X, y[, sample_weight])`


## Our First Estimator

In [17]:
import numpy as np
from sklearn.base import ClassifierMixin, BaseEstimator
from sklearn.utils.validation import check_is_fitted, validate_data
from sklearn.utils.multiclass import check_classification_targets

class MyClassifier(ClassifierMixin, BaseEstimator):
    def fit(self, X, y):
        X, y = validate_data(self, X, y)
        check_classification_targets(y)
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        check_is_fitted(self)
        X = validate_data(self, X, reset=False)
        return np.asarray([self.classes_[0]] * len(X))

    def __sklearn_is_fitted__(self):
        return hasattr(self, "classes_")
    
    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.classifier_tags.poor_score = True
        return tags

In [18]:
from sklearn.utils.estimator_checks import check_estimator
check_estimator(MyClassifier())

[{'estimator': MyClassifier(),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyClassifier(),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyClassifier(),
  'check_name': 'check_estimator_tags_renamed',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyClassifier(),
  'check_name': 'check_valid_tag_types',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyClassifier(),
  'check_name': 'check_estimator_repr',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to

The above check runs all estimator checks against your estimator. You can iteratively fix issues with your estimator until it passes.

``` python
def check_is_fitted(estimator, attributes=None, msg=None, all_or_any=all):
    """Perform is_fitted validation for estimator.

    ...
    """

def validate_data(
    _estimator,
    /,
    X="no_validation",
    y="no_validation",
    reset=True,
    validate_separately=False,
    skip_check_array=False,
    **check_params,
):
    """Validate input data and set or check feature names and counts of the input.

    This helper function should be used in an estimator that requires input
    validation. This mutates the estimator and sets the `n_features_in_` and
    `feature_names_in_` attributes if `reset=True`.

    .. versionadded:: 1.6
    """

def check_classification_targets(y):
    """Ensure that target y is of a non-regression type.

    Only the following target types (as defined in type_of_target) are allowed:
        'binary', 'multiclass', 'multiclass-multioutput',
        'multilabel-indicator', 'multilabel-sequences'
    ...
    """
```

## Run Common Tests on Our Classifier with pytest

``` python
def parametrize_with_checks(
    estimators,
    *,
    legacy: bool = True,
    expected_failed_checks: Callable | None = None,
):
    """Pytest specific decorator for parametrizing estimator checks.

    Checks are categorised into the following groups:

    - API checks: a set of checks to ensure API compatibility with scikit-learn.
      Refer to https://scikit-learn.org/dev/developers/develop.html a requirement of
      scikit-learn estimators.
    - legacy: a set of checks which gradually will be grouped into other categories.

    The `id` of each check is set to be a pprint version of the estimator
    and the name of the check with its keyword arguments.
    This allows to use `pytest -k` to specify which tests to run::

        pytest test_check_estimators.py -k check_estimators_fit_returns_self
    ...
    """
```

In [20]:
from sklearn.utils.estimator_checks import parametrize_with_checks

@parametrize_with_checks([MyClassifier()])
def test_sklearn_compatible_estimator(estimator, check):
    check(estimator)

## A Custom Transformer

In [35]:
from sklearn.base import BaseEstimator, TransformerMixin, OneToOneFeatureMixin
from sklearn.utils.validation import check_is_fitted

class MyTransformer(OneToOneFeatureMixin, TransformerMixin, BaseEstimator):
    def fit(self, X, y):
        validate_data(self, X)
        return self
    
    def __sklearn_is_fitted__(self):
        return True

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.requires_fit = False
        return tags
    
    def transform(self, X):
        X = validate_data(self, X, reset=False)
        return X + 1

In [36]:
from sklearn.utils.estimator_checks import check_estimator
check_estimator(MyTransformer())

[{'estimator': MyTransformer(),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyTransformer(),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyTransformer(),
  'check_name': 'check_estimator_tags_renamed',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyTransformer(),
  'check_name': 'check_valid_tag_types',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': MyTransformer(),
  'check_name': 'check_estimator_repr',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expect

## Transformer in a Pipeline

In [37]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y)

est = make_pipeline(
    StandardScaler(),
    LogisticRegression()
).fit(X_train, y_train)
est[:-1].get_feature_names_out()

array(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'], dtype=object)

In [38]:
est = make_pipeline(
    MyTransformer(),
    StandardScaler(),
    LogisticRegression()
).fit(X_train, y_train)
est[:-1].get_feature_names_out()

array(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'], dtype=object)

In [39]:
est

Pipeline(steps=[('mytransformer', MyTransformer()),
                ('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression())])

## TimedEstimator

In [68]:
from sklearn.base import BaseEstimator, MetaEstimatorMixin, clone
from sklearn.utils.validation import check_is_fitted
from sklearn.utils.metaestimators import available_if
from sklearn.utils import get_tags
import time


class TimedEstimator(MetaEstimatorMixin, BaseEstimator):
    def __init__(self, estimator, verbose=0):
        self.estimator = estimator
        self.verbose = verbose

    def fit(self, X, y, **fit_params):
        start = time.perf_counter()
        self.estimator_ = clone(self.estimator).fit(X, y, **fit_params)
        end = time.perf_counter()
        self.time_ = end - start
        if self.verbose > 0:
            print(self.time_)
        return self

    def __sklearn_clone__(self):
        return TimedEstimator(clone(self.estimator), self.verbose)

    @available_if(lambda self: hasattr(self._get_estimator(), "__getitem__"))
    def __getitem__(self, ind):
        return self._get_estimator().__getitem__(ind)
    
    def __getattr__(self, name):
        if name in ("time_", "estimator_"):
            raise AttributeError(f"{name} is not available here!")
        return getattr(self._get_estimator(), name)

    def _get_estimator(self):
        return getattr(self, "estimator_", None) or self.estimator
    
    def __sklearn_tags__(self):
        return get_tags(self._get_estimator())

In [70]:
from sklearn.utils.estimator_checks import check_estimator
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
check_estimator(TimedEstimator(estimator=LogisticRegression(solver="liblinear")))
check_estimator(TimedEstimator(estimator=LinearRegression()))
check_estimator(TimedEstimator(estimator=StandardScaler()))

[{'estimator': TimedEstimator(estimator=StandardScaler()),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': TimedEstimator(estimator=StandardScaler()),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': TimedEstimator(estimator=StandardScaler()),
  'check_name': 'check_estimator_tags_renamed',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': TimedEstimator(estimator=StandardScaler()),
  'check_name': 'check_valid_tag_types',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': TimedEstimator(estimator=Stand

## TimedEstimator for and in a Pipeline

In [71]:
est = TimedEstimator(
    make_pipeline(
        MyTransformer(),
        StandardScaler(),
        TimedEstimator(LogisticRegression())
    )
).fit(X_train, y_train)

In [72]:
est[-1].time_

0.00678327601053752

## In a GridSearchCV

In [73]:
from sklearn.model_selection import GridSearchCV

est = make_pipeline(
        MyTransformer(),
        StandardScaler(),
        LogisticRegression()
)

grid = {
    "logisticregression__C": [0.01, 0.1, 1, 10, 100]
}

gs = GridSearchCV(estimator=est, param_grid=grid)
gs.fit(X_train, y_train)

GridSearchCV(estimator=Pipeline(steps=[('mytransformer', MyTransformer()),
                                       ('standardscaler', StandardScaler()),
                                       ('logisticregression',
                                        LogisticRegression())]),
             param_grid={'logisticregression__C': [0.01, 0.1, 1, 10, 100]})

In [74]:
gs.best_estimator_

Pipeline(steps=[('mytransformer', MyTransformer()),
                ('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(C=1))])

In [83]:
from sklearn.model_selection import GridSearchCV

est = TimedEstimator(
    make_pipeline(
        MyTransformer(),
        StandardScaler(),
        TimedEstimator(LogisticRegression())
    )
)

grid = {
    "estimator": [
        make_pipeline(
            MyTransformer(),
            StandardScaler(),
            TimedEstimator(LogisticRegression(C=C))
        )
        for C in [0.01, 0.1, 1, 10]
    ]
}

gs = GridSearchCV(estimator=est, param_grid=grid)
gs.fit(X_train, y_train)

GridSearchCV(estimator=TimedEstimator(estimator=Pipeline(steps=[('mytransformer',
                                                                 MyTransformer()),
                                                                ('standardscaler',
                                                                 StandardScaler()),
                                                                ('timedestimator',
                                                                 TimedEstimator(estimator=LogisticRegression()))])),
             param_grid={'estimator': [Pipeline(steps=[('mytransformer',
                                                        MyTransformer()),
                                                       ('standardscaler',
                                                        StandardScaler()),
                                                       ('timedestimator',
                                                        TimedEstimator(est...
                                                        TimedEstimator(estimator=LogisticRegression(C=0.1)))]),
                                       Pipeline(steps=[('mytransformer',
                                                        MyTransformer()),
                                                       ('standardscaler',
                                                        StandardScaler()),
                                                       ('timedestimator',
                                                        TimedEstimator(estimator=LogisticRegression(C=1)))]),
                                       Pipeline(steps=[('mytransformer',
                                                        MyTransformer()),
                                                       ('standardscaler',
                                                        StandardScaler()),
                                                       ('timedestimator',
                                                        TimedEstimator(estimator=LogisticRegression(C=10)))])]})

In [79]:
gs.best_estimator_

TimedEstimator(estimator=Pipeline(steps=[('mytransformer', MyTransformer()),
                                         ('standardscaler', StandardScaler()),
                                         ('timedestimator',
                                          TimedEstimator(estimator=LogisticRegression(C=1)))]))

In [85]:
est.get_params()

{'estimator__memory': None,
 'estimator__steps': [('mytransformer', MyTransformer()),
  ('standardscaler', StandardScaler()),
  ('timedestimator', TimedEstimator(estimator=LogisticRegression()))],
 'estimator__transform_input': None,
 'estimator__verbose': False,
 'estimator__mytransformer': MyTransformer(),
 'estimator__standardscaler': StandardScaler(),
 'estimator__timedestimator': TimedEstimator(estimator=LogisticRegression()),
 'estimator__standardscaler__copy': True,
 'estimator__standardscaler__with_mean': True,
 'estimator__standardscaler__with_std': True,
 'estimator__timedestimator__estimator__C': 1.0,
 'estimator__timedestimator__estimator__class_weight': None,
 'estimator__timedestimator__estimator__dual': False,
 'estimator__timedestimator__estimator__fit_intercept': True,
 'estimator__timedestimator__estimator__intercept_scaling': 1,
 'estimator__timedestimator__estimator__l1_ratio': None,
 'estimator__timedestimator__estimator__max_iter': 100,
 'estimator__timedestimator

In [77]:
from sklearn.model_selection import GridSearchCV

est = TimedEstimator(
    make_pipeline(
        MyTransformer(),
        StandardScaler(),
        TimedEstimator(LogisticRegression())
    )
)

grid = {
    "estimator__timedestimator__estimator__C": [0.01, 0.1, 1, 10, 100]
}

gs = GridSearchCV(estimator=est, param_grid=grid)
gs.fit(X_train, y_train)

GridSearchCV(estimator=TimedEstimator(estimator=Pipeline(steps=[('mytransformer',
                                                                 MyTransformer()),
                                                                ('standardscaler',
                                                                 StandardScaler()),
                                                                ('timedestimator',
                                                                 TimedEstimator(estimator=LogisticRegression()))])),
             param_grid={'estimator__timedestimator__estimator__C': [0.01, 0.1,
                                                                     1, 10,
                                                                     100]})

In [78]:
gs.best_estimator_

TimedEstimator(estimator=Pipeline(steps=[('mytransformer', MyTransformer()),
                                         ('standardscaler', StandardScaler()),
                                         ('timedestimator',
                                          TimedEstimator(estimator=LogisticRegression(C=1)))]))

## Conventions

- `fit` should only get sample aligned data in `fit_params`
    - everything else should go through `__init__`
- `__init__` doesn't set anything other than the parameters passed to it
- `obj.attr` is set through `__init__` and `set_params`
- `obj.attr_` is set during fit and counts as public API
- `obj._attr` is private
- `n_features_in_`, `feature_names_in_`, and `get_feature_names_out()`

## Estimator Tags

``` python
@dataclass(slots=True)
class InputTags:
    """Tags for the input data.

    Parameters
    ----------
    one_d_array : bool, default=False
        Whether the input can be a 1D array.

    two_d_array : bool, default=True
        Whether the input can be a 2D array. Note that most common
        tests currently run only if this flag is set to ``True``.

    three_d_array : bool, default=False
        Whether the input can be a 3D array.

    sparse : bool, default=False
        Whether the input can be a sparse matrix.

    categorical : bool, default=False
        Whether the input can be categorical.

    string : bool, default=False
        Whether the input can be an array-like of strings.

    dict : bool, default=False
        Whether the input can be a dictionary.

    positive_only : bool, default=False
        Whether the estimator requires positive X.

    allow_nan : bool, default=False
        Whether the estimator supports data with missing values encoded as `np.nan`.

    pairwise : bool, default=False
        This boolean attribute indicates whether the data (`X`),
        :term:`fit` and similar methods consists of pairwise measures
        over samples rather than a feature representation for each
        sample.  It is usually `True` where an estimator has a
        `metric` or `affinity` or `kernel` parameter with value
        'precomputed'. Its primary purpose is to support a
        :term:`meta-estimator` or a cross validation procedure that
        extracts a sub-sample of data intended for a pairwise
        estimator, where the data needs to be indexed on both axes.
        Specifically, this tag is used by
        `sklearn.utils.metaestimators._safe_split` to slice rows and
        columns.

        Note that if setting this tag to ``True`` means the estimator can take only
        positive values, the `positive_only` tag must reflect it and also be set to
        ``True``.
    """

    one_d_array: bool = False
    two_d_array: bool = True
    three_d_array: bool = False
    sparse: bool = False
    categorical: bool = False
    string: bool = False
    dict: bool = False
    positive_only: bool = False
    allow_nan: bool = False
    pairwise: bool = False


@dataclass(slots=True)
class TargetTags:
    """Tags for the target data.

    Parameters
    ----------
    required : bool
        Whether the estimator requires y to be passed to `fit`,
        `fit_predict` or `fit_transform` methods. The tag is ``True``
        for estimators inheriting from `~sklearn.base.RegressorMixin`
        and `~sklearn.base.ClassifierMixin`.

    one_d_labels : bool, default=False
        Whether the input is a 1D labels (y).

    two_d_labels : bool, default=False
        Whether the input is a 2D labels (y).

    positive_only : bool, default=False
        Whether the estimator requires a positive y (only applicable
        for regression).

    multi_output : bool, default=False
        Whether a regressor supports multi-target outputs or a classifier supports
        multi-class multi-output.

        See :term:`multi-output` in the glossary.

    single_output : bool, default=True
        Whether the target can be single-output. This can be ``False`` if the
        estimator supports only multi-output cases.
    """

    required: bool
    one_d_labels: bool = False
    two_d_labels: bool = False
    positive_only: bool = False
    multi_output: bool = False
    single_output: bool = True


@dataclass(slots=True)
class TransformerTags:
    """Tags for the transformer.

    Parameters
    ----------
    preserves_dtype : list[str], default=["float64"]
        Applies only on transformers. It corresponds to the data types
        which will be preserved such that `X_trans.dtype` is the same
        as `X.dtype` after calling `transformer.transform(X)`. If this
        list is empty, then the transformer is not expected to
        preserve the data type. The first value in the list is
        considered as the default data type, corresponding to the data
        type of the output when the input data type is not going to be
        preserved.
    """

    preserves_dtype: list[str] = field(default_factory=lambda: ["float64"])


@dataclass(slots=True)
class ClassifierTags:
    """Tags for the classifier.

    Parameters
    ----------
    poor_score : bool, default=False
        Whether the estimator fails to provide a "reasonable" test-set
        score, which currently for classification is an accuracy of
        0.83 on ``make_blobs(n_samples=300, random_state=0)``. The
        datasets and values are based on current estimators in scikit-learn
        and might be replaced by something more systematic.

    multi_class : bool, default=True
        Whether the classifier can handle multi-class
        classification. Note that all classifiers support binary
        classification. Therefore this flag indicates whether the
        classifier is a binary-classifier-only or not.

        See :term:`multi-class` in the glossary.

    multi_label : bool, default=False
        Whether the classifier supports multi-label output: a data point can
        be predicted to belong to a variable number of classes.

        See :term:`multi-label` in the glossary.
    """

    poor_score: bool = False
    multi_class: bool = True
    multi_label: bool = False


@dataclass(slots=True)
class RegressorTags:
    """Tags for the regressor.

    Parameters
    ----------
    poor_score : bool, default=False
        Whether the estimator fails to provide a "reasonable" test-set
        score, which currently for regression is an R2 of 0.5 on
        ``make_regression(n_samples=200, n_features=10,
        n_informative=1, bias=5.0, noise=20, random_state=42)``. The
        dataset and values are based on current estimators in scikit-learn
        and might be replaced by something more systematic.
    """

    poor_score: bool = False


@dataclass(slots=True)
class Tags:
    """Tags for the estimator.

    See :ref:`estimator_tags` for more information.

    Parameters
    ----------
    estimator_type : str or None
        The type of the estimator. Can be one of:
        - "classifier"
        - "regressor"
        - "transformer"
        - "clusterer"
        - "outlier_detector"
        - "density_estimator"

    target_tags : :class:`TargetTags`
        The target(y) tags.

    transformer_tags : :class:`TransformerTags` or None
        The transformer tags.

    classifier_tags : :class:`ClassifierTags` or None
        The classifier tags.

    regressor_tags : :class:`RegressorTags` or None
        The regressor tags.

    array_api_support : bool, default=False
        Whether the estimator supports Array API compatible inputs.

    no_validation : bool, default=False
        Whether the estimator skips input-validation. This is only meant for
        stateless and dummy transformers!

    non_deterministic : bool, default=False
        Whether the estimator is not deterministic given a fixed ``random_state``.

    requires_fit : bool, default=True
        Whether the estimator requires to be fitted before calling one of
        `transform`, `predict`, `predict_proba`, or `decision_function`.

    _skip_test : bool, default=False
        Whether to skip common tests entirely. Don't use this unless
        you have a *very good* reason.

    input_tags : :class:`InputTags`
        The input data(X) tags.
    """

    estimator_type: str | None
    target_tags: TargetTags
    transformer_tags: TransformerTags | None = None
    classifier_tags: ClassifierTags | None = None
    regressor_tags: RegressorTags | None = None
    array_api_support: bool = False
    no_validation: bool = False
    non_deterministic: bool = False
    requires_fit: bool = True
    _skip_test: bool = False
    input_tags: InputTags = field(default_factory=InputTags)
```

## Metadata Routing

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    GridSearchCV, cross_validate,
    GroupKFold, KFold
)

rng = np.random.default_rng(42)
X = rng.random(size=(100, 10))
y = rng.integers(0, 2, size=100)
groups = rng.integers(0, 10, size=100)
revenue = rng.random(100)
sample_weight = rng.random(100)

In [ ]:
import sklearn
sklearn.set_config(enable_metadata_routing=True)

outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=5)
model = LogisticRegression().set_fit_request(sample_weight=True).set_score_request(sample_weight=True)
grid = {"C": [1, 10]}
search = GridSearchCV(model, grid, cv=inner_cv, n_jobs=1)
outer_eval = cross_validate(
    search,
    X,
    y=y,
    cv=outer_cv,
    n_jobs=1,
    params={"groups": groups, "sample_weight":sample_weight},
)

## Upcoming Features

- metadata routing through `get_metadata_routing()` and `set_{method}_request()` [SLEP#6](https://scikit-learn-enhancement-proposals.readthedocs.io/en/latest/slep006/proposal.html), [#22893](https://github.com/scikit-learn/scikit-learn/issues/22893)
    - At this point you should implement it
    - [User guide for users](https://scikit-learn.org/stable/metadata_routing.html) and [documentation for developers](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_metadata_routing.html)
- standardized parameter validation through `validate_params` [#23462](https://github.com/scikit-learn/scikit-learn/issues/23462): We do it internally, but your estimators are not tested against it and our utils are private for now.
- Callbacks: [#28760](https://github.com/scikit-learn/scikit-learn/pull/28760)

## More Details/Further Reading

- [https://scikit-learn.org/dev/developers/develop.html](https://scikit-learn.org/dev/developers/develop.html)
- `sklearn/base.py`
- `sklearn/metaestimators.py`
- `sklearn/utils/metaestimators.py`
- `sklearn/utils/validation.py`

## Questions?
### Thank you!